In [28]:
import numpy as np
from pathlib import Path

In [37]:
def update_equation(D,A=None,pc=None,pr=None,order='ST'):
    """ 
    update_equation updates the probability(ies) to be degraded, D, by one time step, 
    given the interaction matrix A, collapse rate pc and recovery rate pr

    Parameters
    ---
    D : numpy.array
        The initial array of degraded probabilities
    A : numpy.array
        The interaction matrix (i.e. adjacency matrix), must be of shape (len(D), len(D)). 
        No self loops, i.e. all diagonal elements should be 0.
        Default is np.zeros([len(D)]*2) i.e. the empty graph, such that all Markov chains are disconnected.
    pc : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]. 
        If none supplied, default is 1/2 for all Markov chains
    pr : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]
        If none supplied default is tr = 10tc where tr is the recovery time and 
        tc the collapse time with pc=1/(tc+1)
    order : str
        The order of the interaction matrix, 
        if 'ST' then the interaction matrix is indexed as A[source, target]
        if 'TS' then the interaction matrix is indexed as A[target, source]
        Default is 'ST'
        
    Returns
    ---
    D_out : numpy.array

    Examples
    ---
    For an initial vector of zeros, regardless of the interaction matrix, D_out will always be pc
    >>> update_equation(np.zeros(2), A = np.array( [[0, -2], [4, 0]] ), pc = np.array([0.2, 0.9]), order='ST')
    np.array([0.2, 0.9]) 
    
    For an initial vector of ones, regardless of the interaction matrix, D_out will always be pr
    >>> update_equation(np.ones(2), A = np.array( [[0, -2], [4, 0]] ), pc = np.array([0.2, 0.9]), pr = np.array([0.5, 0.5]), order='ST')
    np.array([0.5, 0.5])

    For non-trivial initial D:
    >>> update_equation(np.array([0, 1]), A = np.array( [[0, -2], [4, 0]] ), pc = np.array([0.2, 0.9]), pr = np.array([0.5, 0.5]), order='ST')
    """
    if A is None:
        A = np.zeros([len(D)]*2)
    elif order == 'ST':
        A = A.transpose()
    if pc is None:
        pc = np.ones(len(D))/2
    if pr is None:
        pr = 1/(10/pc - 9)

    ### Decomposing interaction matrix as A = A_pos + A_neg, 
        ### where A_pos contains all positive elements of A, while A_neg contains all negatives
    A_pos = np.maximum(A,0)
    A_neg = np.maximum(-A,0)
    d_pos = np.dot(A_pos,D)
    d_neg = np.dot(A_neg,D)

    return (1+d_pos)*pc*(1-D)/(1+d_neg) + (1-pr)*D

def temp_response(T,T_lim,beta):
    """ 
    
    """
    return 1/(1+np.exp(beta*(T_lim-T)))

In [40]:
help(update_equation)

Help on function update_equation in module __main__:

update_equation(D, A=None, pc=None, pr=None, order='ST')
    update_equation updates the probability(ies) to be degraded, D, by one time step,
    given the interaction matrix A, collapse rate pc and recovery rate pr

    Parameters
    ---
    D : numpy.array
        The initial array of degraded probabilities
    A : numpy.array
        The interaction matrix (i.e. adjacency matrix), must be of shape (len(D), len(D)).
        Default is np.zeros([len(D)]*2) i.e. the empty graph, such that all Markov chains are disconnected.
    pc : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1].
        If none supplied, default is 1/2 for all Markov chains
    pr : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]
        If none supplied default is tr = 10tc where tr is the recovery time and
        tc the collapse time with 

In [22]:
A = np.random.randint(-10,10,size=(3,3))
print(A)

[[ 4 -5  7]
 [-3  3  2]
 [-8 -3  2]]


In [23]:
print(A_pos)
print(A_neg)

[[4 0 7]
 [0 3 2]
 [0 0 2]]
[[ 0 -5  0]
 [-3  0  0]
 [-8 -3  0]]


# Reading generated interaction matrices

In [25]:
folder = "../Data/Interaction matrices/"
no_samples = 500
A = [np.loadtxt(folder+str(r)+'.csv') for r in range(no_samples)]

In [41]:
print(A[0])

[[ 0.          0.          0.          0.          0.          0.
   0.          0.          0.        ]
 [ 0.          0.          1.33237156  2.02383507  0.          0.
   0.          0.          1.04022105]
 [ 0.          0.          0.          3.41147869  0.          0.
   0.          0.          0.        ]
 [ 0.         -1.22198619  0.          0.          3.97785539  1.2383388
   1.38920816  0.84196562 -6.1114675 ]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.        ]
 [ 5.74695746  0.          0.          0.          0.          2.21026244
   0.          2.75967476  0.        ]
 [ 0.          0.          0.          1.09106138  0.          0.
   0.          0.          1.64277193]
 [ 0.          0.          0.          5.54865669  0.          0.
   0.          2.7319117   0.        ]]


In [33]:
print(A[0].transpose() )

[[ 0.          0.          0.          0.          0.          0.
   5.74695746  0.          0.        ]
 [ 0.          0.          0.         -1.22198619  0.          0.
   0.          0.          0.        ]
 [ 0.          1.33237156  0.          0.          0.          0.
   0.          0.          0.        ]
 [ 0.          2.02383507  3.41147869  0.          0.          0.
   0.          1.09106138  5.54865669]
 [ 0.          0.          0.          3.97785539  0.          0.
   0.          0.          0.        ]
 [ 0.          0.          0.          1.2383388   0.          0.
   2.21026244  0.          0.        ]
 [ 0.          0.          0.          1.38920816  0.          0.
   0.          0.          0.        ]
 [ 0.          0.          0.          0.84196562  0.          0.
   2.75967476  0.          2.7319117 ]
 [ 0.          1.04022105  0.         -6.1114675   0.          0.
   0.          1.64277193  0.        ]]


In [29]:
txt = Path(folder+'README.txt').read_text()
print(txt)

Index ordering of the tipping elements:
{'REEF': 0, 'ASI': 1, 'PERM': 2, 'AMOC': 3, 'WAM': 4, 'AMAZ': 5, 'ENSO': 6, 'WAIS': 7, 'GIS': 8}

Random seed 1850201420152350
(use np.random.default_rng(seed=1850201420152350))


In [42]:
D = np.ones(9)
D_new = np.dot(np.transpose(A[0]),D)
print(D_new)

a_pos = np.maximum(A[0],0)
a_neg = np.maximum(-A[0],0)

[ 5.74695746 -1.22198619  1.33237156 12.07503183  3.97785539  3.44860125
  1.38920816  6.33355208 -3.42847452]


In [43]:
print(a_pos)
print(a_neg)

[[0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]
 [0.         0.         1.33237156 2.02383507 0.         0.
  0.         0.         1.04022105]
 [0.         0.         0.         3.41147869 0.         0.
  0.         0.         0.        ]
 [0.         0.         0.         0.         3.97785539 1.2383388
  1.38920816 0.84196562 0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]
 [5.74695746 0.         0.         0.         0.         2.21026244
  0.         2.75967476 0.        ]
 [0.         0.         0.         1.09106138 0.         0.
  0.         0.         1.64277193]
 [0.         0.         0.         5.54865669 0.         0.
  0.         2.7319117  0.        ]]
[[0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]
 [0.         0.         

# Testing

In [38]:
D = np.zeros(9)
D_out = update_equation(D,A=A[0])

In [39]:
print(D_out)

[0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5]
